In [3]:
import json
import time


# 1. COMMAND MODEL (Слой Записи / Event Store)
# Здесь мы используем паттерн Event Sourcing. Мы храним не текущее состояние,
# а ИСТОРИЮ ИЗМЕНЕНИЙ (прямо как в LSM-Tree WAL или Kafka).

event_store_log = "event_sourcing_log.txt"

def write_command(user_id: int, action: str, payload: dict):
    """
    Быстрая операция ЗАПИСИ (O(1)). 
    Никаких B-Деревьев, никаких блокировок (Row Locks). Просто Append в конец лога.
    """
    event = {
        "timestamp": time.time(),
        "user_id": user_id,
        "action": action,
        "payload": payload
    }
    with open(event_store_log, "a") as f:
        f.write(json.dumps(event) + "\n")
    print(f"[Command] -> Факт записан в лог: User {user_id} сделал '{action}'")

# 2. QUERY MODEL (Слой Чтения / Материализованное представление)
# Это база данных, оптимизированная строго под нужды фронтенда (O(1) на чтение).
# Она строится асинхронно, читая события из Лога.

class UserBalanceReadModel:
    def __init__(self):
        # Наша In-Memory "База" для быстрых чтений (например, кэш Redis)
        self.balances = {}

    def handle_event(self, event: dict):
        """Интерпретатор событий (Проектор)"""
        uid = event["user_id"]
        action = event["action"]
        payload = event["payload"]

        # Инициализируем юзера при первой встрече
        if uid not in self.balances:
            self.balances[uid] = 0

        # Обновляем материализованное представление
        if action == "USER_CREATED":
            self.balances[uid] = payload.get("initial_balance", 0)
        elif action == "FUNDS_ADDED":
            self.balances[uid] += payload.get("amount", 0)
        elif action == "FUNDS_WITHDRAWN":
            self.balances[uid] -= payload.get("amount", 0)

    def get_balance(self, user_id: int) -> int:
        """Сверхбыстрое чтение O(1). Тот самый Query."""
        return self.balances.get(user_id, 0)
        
    def rebuild_from_scratch(self, log_filepath: str):
        """
        Если наша кэш-база упала и сгорела, Мы просто берем неподвижный лог (Кафку) 
        и перематываем события с начала времен.
        """
        print("\n[Architecture] Read-База отвалилилась. Запускаем пересчет (Replay/Backfill)...")
        self.balances.clear()
        
        start_time = time.time()
        with open(log_filepath, "r") as f:
            for line in f:
                event = json.loads(line)
                self.handle_event(event) # Прогоняем тот же самый код логики!
                
        print(f"[Architecture] База восстановлена за {time.time() - start_time:.4f} сек!")


# 3. ДЕМОНСТРАЦИЯ РАБОТЫ (System Design in Action)
if __name__ == "__main__":
    # Очищаем старый лог для чистоты эксперимента
    open(event_store_log, "w").close()

    print("1. ГЕНЕРАЦИЯ СОБЫТИЙ")
    # Система Записи принимает команды пользователей со скоростью диска
    write_command(1, "USER_CREATED", {"initial_balance": 100})
    write_command(2, "USER_CREATED", {"initial_balance": 50})
    write_command(1, "FUNDS_ADDED", {"amount": 500})
    write_command(2, "FUNDS_WITHDRAWN", {"amount": 10})
    write_command(1, "FUNDS_WITHDRAWN", {"amount": 50})

    print("2. НОРМАЛЬНЫЙ РЕЖИМ (Обработка брокером)")
    read_model = UserBalanceReadModel()
    
    # Эмуляция того, как микросервис асинхронно вычитывает новые события и обновляет базу
    with open(event_store_log, "r") as f:
        for _line in f:
            read_model.handle_event(json.loads(_line))
            
    print(f"[Query] Баланс User 1: {read_model.get_balance(1)} руб. (Ожидаем 550)")
    print(f"[Query] Баланс User 2: {read_model.get_balance(2)} руб. (Ожидаем 40)")
    
    print("3. ИСПЫТАНИЕ KAPPA АРХИТЕКТУРЫ (ИЗМЕНЕНИЕ БИЗНЕС ЛОГИКИ)")
    #Бизнес сказал: 'Мы ошиблись, за каждое пополнение нужно было брать комиссию 10%!'
    #Как это делают в классической БД? Пишут страшный скрипт UPDATE, который лочит БД на сутки.
    #Как это делаем мы? Просто меняем функцию handle_event и делаем Replay
    
    #переопределяем логику проектора (имитируем деплой новой версии кода)
    def new_handle_event(self, event):
        uid = event["user_id"]
        action = event["action"]
        payload = event["payload"]
        if uid not in self.balances: self.balances[uid] = 0
        
        if action == "USER_CREATED":
            self.balances[uid] = payload.get("initial_balance", 0)
        elif action == "FUNDS_ADDED":
            # НОВАЯ ЛОГИКА: Вычитаем 10% комиссии
            amount = payload.get("amount", 0)
            self.balances[uid] += (amount * 0.9) 
        elif action == "FUNDS_WITHDRAWN":
            self.balances[uid] -= payload.get("amount", 0)

    # Применяем 'патч' к классу
    UserBalanceReadModel.handle_event = new_handle_event

    # ВОССТАНАВЛИВАЕМ ДАННЫЕ ПРОКРУТКОЙ ЛОГА С НУЛЯ (Replay)
    read_model.rebuild_from_scratch(event_store_log)
    
    print(f"[Query] НОВЫЙ Баланс User 1: {read_model.get_balance(1)} руб. (Начислили 500 - 10% = 450. Итог: 100+450-50 = 500!)")
    print("Вся историческая справедливость восстановлена без единого UPDATE в базе!")


1. ГЕНЕРАЦИЯ СОБЫТИЙ
[Command] -> Факт записан в лог: User 1 сделал 'USER_CREATED'
[Command] -> Факт записан в лог: User 2 сделал 'USER_CREATED'
[Command] -> Факт записан в лог: User 1 сделал 'FUNDS_ADDED'
[Command] -> Факт записан в лог: User 2 сделал 'FUNDS_WITHDRAWN'
[Command] -> Факт записан в лог: User 1 сделал 'FUNDS_WITHDRAWN'
2. НОРМАЛЬНЫЙ РЕЖИМ (Обработка брокером)
[Query] Баланс User 1: 550 руб. (Ожидаем 550)
[Query] Баланс User 2: 40 руб. (Ожидаем 40)
3. ИСПЫТАНИЕ KAPPA АРХИТЕКТУРЫ (ИЗМЕНЕНИЕ БИЗНЕС ЛОГИКИ)

[Architecture] Read-База отвалилилась. Запускаем пересчет (Replay/Backfill)...
[Architecture] База восстановлена за 0.0001 сек!
[Query] НОВЫЙ Баланс User 1: 500.0 руб. (Начислили 500 - 10% = 450. Итог: 100+450-50 = 500!)
Вся историческая справедливость восстановлена без единого UPDATE в базе!
